# Exploracion de bases de datos oficiales disponibles
## Notebook para explorar los archivos sobre circuitos electorlaes disponibles en sitios oficiales y evaluar su estado actual
## Genera una tabla con los circuitos por provincia segun el mapa, su coincidencias y sus diferencias con el mapa actual y un resumen de totales.

In [1]:
import json
import glob
import pandas as pd
import geopandas as gpd
from pathlib import Path
from IPython.display import display

In [2]:
path = "/Users/francogaleano/Documents/Elecciones_Argentina/2025_GENERAL/resultados2025.csv"

# Load only the necessary columns to reduce memory usage
cols = ['distrito_id', 'distrito_nombre', 'seccion_id', 'seccion_nombre',
        'circuito_id', 'circuito_nombre', 'mesa_id', 'mesa_electores']

df = pd.read_csv(path, usecols=cols)

/var/folders/70/cljtltjd17192m_cpsszx5sr0000gn/T/ipykernel_20446/3869605385.py:7: DtypeWarning: Columns (10,11) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path, usecols=cols)


In [3]:
# Drop duplicate rows, then group by circuit and sum the number of voters
df_electores = (
    df
    .drop_duplicates()
    .groupby(['distrito_id', 'distrito_nombre', 'seccion_id', 'seccion_nombre',
              'circuito_id', 'circuito_nombre'], as_index=False)
    .agg(cantidad_electores=('mesa_electores', 'sum'))
)

# Free memory from the raw dataframe
del df

df_electores.head()

,distrito_id,distrito_nombre,seccion_id,seccion_nombre,circuito_id,circuito_nombre,cantidad_electores
0,1,CIUDAD AUTÓNOMA DE BUENOS AIRES,1,COMUNA 1,1,1,10283
1,1,CIUDAD AUTÓNOMA DE BUENOS AIRES,1,COMUNA 1,2,2,9338
2,1,CIUDAD AUTÓNOMA DE BUENOS AIRES,1,COMUNA 1,3,3,3344
3,1,CIUDAD AUTÓNOMA DE BUENOS AIRES,1,COMUNA 1,4,4,2581
4,1,CIUDAD AUTÓNOMA DE BUENOS AIRES,1,COMUNA 1,5,5,13252


In [4]:
# Read GeoJSON file
pba_gdf = gpd.read_file("2025/circuito-electorales-pba.geojson")

# Quick overview
print("Shape:", pba_gdf.shape)
print("\nColumns:\n", pba_gdf.columns.tolist())
print("\nFirst rows:")
print(pba_gdf.head())

Shape: (1147, 9)

Columns:
 ['gid', 'distrito', 'provincia', 'departamen', 'cabecera', 'circuito', 'indec_p', 'indec_d', 'geometry']

First rows:
     gid distrito     provincia     departamen               cabecera  \
0  64876       02  Buenos Aires           Azul                   Azul   
1  64888       02  Buenos Aires       Baradero               Baradero   
2  64891       02  Buenos Aires        Bolivar  San Carlos de Bolivar   
3  64893       02  Buenos Aires        Bolivar  San Carlos de Bolivar   
4  64962       02  Buenos Aires  Benito Juarez          Benito Juarez   

  circuito indec_p indec_d                                           geometry  
0     0063      02     049  MULTIPOLYGON (((-59.43389 -37.00628, -59.43538...  
1     0111      02     070  MULTIPOLYGON (((-59.40638 -34.05963, -59.40671...  
2     0125      02     105  MULTIPOLYGON (((-61.04872 -36.35132, -61.05215...  
3     0128      02     105  MULTIPOLYGON (((-60.96984 -35.92613, -60.96799...  
4     0436     

In [5]:

# --- Filter df_electores to PBA only (distrito_id == 2) ---
pba_electores = df_electores[df_electores['distrito_id'] == 2].copy()

# Normalize: strip leading zeros and uppercase (handles letters like 666A, 207L)
# e.g. "0063" -> "63", "0666A" -> "666A", "668b" -> "668B"
pba_electores['circuito_key'] = pba_electores['circuito_id'].astype(str).str.lstrip('0').str.upper()
pba_gdf['circuito_key']       = pba_gdf['circuito'].astype(str).str.lstrip('0').str.upper()

# --- Compare ---
electores_set = set(pba_electores['circuito_key'])
geo_set       = set(pba_gdf['circuito_key'])

matched   = electores_set & geo_set       # circuitos en ambos
missing   = electores_set - geo_set       # en electores pero sin geometría
extra_geo = geo_set - electores_set       # en geo pero sin datos electorales

print(f"Circuitos en df_electores (PBA):  {len(electores_set)}")
print(f"Circuitos en GeoJSON:             {len(geo_set)}")
print(f"Circuitos matcheados:             {len(matched)}")
print(f"Faltantes en GeoJSON:             {len(missing)}")
print(f"En GeoJSON pero sin datos:        {len(extra_geo)}")

# --- Detail of missing circuits with their voter count ---
df_missing = (
    pba_electores[pba_electores['circuito_key'].isin(missing)]
    [['seccion_id', 'seccion_nombre', 'circuito_id', 'circuito_nombre', 'circuito_key', 'cantidad_electores']]
    .sort_values(['seccion_nombre', 'cantidad_electores'], ascending=[True, False])
)

print(f"\nCircuitos sin geometría y su cantidad de electores:")
print(df_missing.to_string(index=False))
print(f"\nTotal electores sin geometría: {df_missing['cantidad_electores'].sum():,}")
print(f"Total electores PBA:           {pba_electores['cantidad_electores'].sum():,}")
print(f"% sin cobertura:               {df_missing['cantidad_electores'].sum() / pba_electores['cantidad_electores'].sum() * 100:.2f}%")

Circuitos en df_electores (PBA):  1047
Circuitos en GeoJSON:             1145
Circuitos matcheados:             1042
Faltantes en GeoJSON:             5
En GeoJSON pero sin datos:        103

Circuitos sin geometría y su cantidad de electores:
 seccion_id     seccion_nombre circuito_id circuito_nombre circuito_key  cantidad_electores
          3    ALMIRANTE BROWN       0022C             22C          22C               29872
         52 GENERAL SAN MARTÍN       0388B            388B         388B               35267
         52 GENERAL SAN MARTÍN       0388A            388A         388A               32760
         52 GENERAL SAN MARTÍN       0388C            388C         388C               17405
         91               PUÁN       00779             779          779                 425

Total electores sin geometría: 115,729
Total electores PBA:           13,349,014
% sin cobertura:               0.87%


In [6]:
path_padron = "/Users/francogaleano/Desktop/LSE/capstone/bases_registro_infractores2025_2/G 2025 02 PADINF con fecnac (anonimizado).txt"

# --- Load: try tab separator, fallback to semicolon if needed ---
df_padron = pd.read_csv(
    path_padron,
    sep="|",
    dtype=str,
    encoding="utf-8",
    on_bad_lines="skip",
    engine="python",
    usecols=['TX_CIRC_NUMERO']
)

# Quick check
print("Shape:", df_padron.shape)
print(df_padron.head(10))

Shape: (13353974, 1)
  TX_CIRC_NUMERO
0           370G
1           370G
2           370G
3           370G
4           370G
5           370G
6           370G
7           370G
8           370G
9           370G


In [8]:
# --- Count voters per circuit ---
df_padron_agg = (
    df_padron
    .groupby('TX_CIRC_NUMERO', as_index=False)
    .size()
    .rename(columns={'size': 'cantidad_electores'})
)

# --- Normalize circuit key: strip leading zeros, uppercase ---
df_padron_agg['circuito_key'] = df_padron_agg['TX_CIRC_NUMERO'].str.strip().str.lstrip('0').str.upper()

del df_padron

# --- Compare against GeoJSON ---
padron_set = set(df_padron_agg['circuito_key'])
geo_set    = set(pba_gdf['circuito_key'])

matched   = padron_set & geo_set
missing   = padron_set - geo_set
extra_geo = geo_set - padron_set

print(f"Circuitos en padrón:          {len(padron_set)}")
print(f"Circuitos en GeoJSON:         {len(geo_set)}")
print(f"Circuitos matcheados:         {len(matched)}")
print(f"Faltantes en GeoJSON:         {len(missing)}")
print(f"En GeoJSON pero sin padrón:   {len(extra_geo)}")

# --- Detail of missing circuits ---
df_missing_padron = (
    df_padron_agg[df_padron_agg['circuito_key'].isin(missing)]
    [['TX_CIRC_NUMERO', 'circuito_key', 'cantidad_electores']]
    .sort_values('cantidad_electores', ascending=False)
)

print(f"\nCircuitos sin geometría:")
print(df_missing_padron.to_string(index=False))
print(f"\nTotal electores sin geometría: {df_missing_padron['cantidad_electores'].sum():,}")
print(f"Total electores padrón:        {df_padron_agg['cantidad_electores'].sum():,}")
print(f"% sin cobertura:               {df_missing_padron['cantidad_electores'].sum() / df_padron_agg['cantidad_electores'].sum() * 100:.2f}%")

Circuitos en padrón:          1110
Circuitos en GeoJSON:         1145
Circuitos matcheados:         1105
Faltantes en GeoJSON:         5
En GeoJSON pero sin padrón:   40

Circuitos sin geometría:
TX_CIRC_NUMERO circuito_key  cantidad_electores
          388B         388B               35227
          388A         388A               32774
           22C          22C               29860
          388C         388C               17410
           779          779                 428

Total electores sin geometría: 115,699
Total electores padrón:        13,353,974
% sin cobertura:               0.87%


In [9]:
print(pba_gdf[pba_gdf['circuito_key'].isin(['388', '22'])])

       gid distrito     provincia          departamen            cabecera  \
841  65611       02  Buenos Aires  General San Martin  General San Martin   
899  65669       02  Buenos Aires     Almirante Brown     Almirante Brown   

    circuito indec_p indec_d  \
841     0388      02     371   
899     0022      02     028   

                                              geometry circuito_key  
841  MULTIPOLYGON (((-58.54787 -34.57603, -58.54798...          388  
899  MULTIPOLYGON (((-58.35118 -34.87823, -58.35097...           22  
